<a href="https://colab.research.google.com/github/westbullyboi/UVR5_NO_UI_ja/blob/main/UVR5_NO_UI_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🎵 UVR5 NO UI 🎵**

Colab作成者：**[AI HUB](https://discord.gg/aihub)** コミュニティの **[Not Eddy (Spanish Mod)](http://discord.com/users/274566299349155851)**

<br>**このColabは、[beveradb の python-audio-separator](https://github.com/karaokenerds/python-audio-separator) リポジトリをベースに作成された CLI 形式の Colab です。<br>
気に入っていただけたら、[GitHub](https://github.com/Eddycrack864/UVR5-NO-UI) のリポジトリにスターをお願いします。**<br>

<br>本家 UVR5 プロジェクトへの寄付はこちらから：<br>
[!["Buy Me A Coffee"](https://www.buymeacoffee.com/assets/img/custom_images/orange_img.png)](https://www.buymeacoffee.com/uvr5)

<br>**使い方：https://github.com/Eddycrack864/UVR5-NO-UI#how-to-use**

![](https://count.nett.moe/get/uvr5_no_ui_colab/img?theme=rule34)

In [ ]:
#@title # **インストール**
#@markdown ####約2分かかります
from IPython.display import clear_output
import subprocess
import os

colab_path = "/content"
kaggle_path = "/kaggle/working"

if os.path.exists(colab_path):
  print("Colab ノートブックへようこそ")
  from google.colab import drive
  drive.mount('/content/drive', force_remount=True)
  path = "/content"
elif os.path.exists(kaggle_path):
  print("Kaggle ノートブックへようこそ")
  path = "/kaggle/working"

!pip install "audio-separator[gpu]==0.44.2"
subprocess.run(["pip", "install", "demucs"])
!pip install aria2
!pip install yt_dlp
!mkdir models
!mkdir temp
!aria2c https://huggingface.co/Eddycrack864/Drumsep/resolve/main/modelo_final.th -o models/drumsep.th
!apt-get update
!pip install "onnxruntime-gpu[cuda]==1.26.0"

clear_output()
print('インストールが完了しました！')

# <small> **分離方法を選択してください：**

### ***BS-Roformer***

In [ ]:
#@markdown #**分離実行！（BS-Roformer / Mel Band Roformer 専用）**
import os
import glob
import yt_dlp

def downloader(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

def checker(url):
    return "http" in url

def uvr_cli(audio_input, output_folder, model, output_format, segment_size, overlap, batch_size, override_model_segment_size, use_autocast, extensions):
    found_files = []

    dictmodel = {
      'BS-Roformer-Viperx-1297': 'model_bs_roformer_ep_317_sdr_12.9755.ckpt',
      'BS-Roformer-Viperx-1296': 'model_bs_roformer_ep_368_sdr_12.9628.ckpt',
      'BS-Roformer-Viperx-1053': 'model_bs_roformer_ep_937_sdr_10.5309.ckpt',
      'Mel-Roformer-Viperx-1143': 'model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt',
      'BS-Roformer-De-Reverb': 'deverb_bs_roformer_8_384dim_10depth.ckpt',
      'Mel-Roformer-Crowd-Aufr33-Viperx': 'mel_band_roformer_crowd_aufr33_viperx_sdr_8.7144.ckpt',
      'Mel-Roformer-Denoise-Aufr33': 'denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt',
      'Mel-Roformer-Denoise-Aufr33-Aggr' : 'denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt',
      'MelBand Roformer | Denoise-Debleed by Gabox' : 'mel_band_roformer_denoise_debleed_gabox.ckpt',
      'Mel-Roformer-Karaoke-Aufr33-Viperx': 'mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt',
      'MelBand Roformer | Karaoke by Gabox' : 'mel_band_roformer_karaoke_gabox.ckpt',
      'MelBand Roformer | Karaoke V2 by Gabox' : 'mel_band_roformer_karaoke_gabox_v2.ckpt',
      'MelBand Roformer | Karaoke by becruily' : 'mel_band_roformer_karaoke_becruily.ckpt',
      'MelBand Roformer | Vocals by Kimberley Jensen' : 'vocals_mel_band_roformer.ckpt',
      'MelBand Roformer Kim | FT by unwa' : 'mel_band_roformer_kim_ft_unwa.ckpt',
      'MelBand Roformer Kim | FT 2 by unwa' : 'mel_band_roformer_kim_ft2_unwa.ckpt',
      'MelBand Roformer Kim | FT 2 Bleedless by unwa' : 'mel_band_roformer_kim_ft2_bleedless_unwa.ckpt',
      'MelBand Roformer Kim | FT 3 by unwa' : 'mel_band_roformer_kim_ft3_unwa.ckpt',
      'MelBand Roformer Kim | Inst V1 by Unwa' : 'melband_roformer_inst_v1.ckpt',
      'MelBand Roformer Kim | Inst V1 Plus by Unwa' : 'melband_roformer_inst_v1_plus.ckpt',
      'MelBand Roformer Kim | Inst V1 (E) by Unwa' : 'melband_roformer_inst_v1e.ckpt',
      'MelBand Roformer Kim | Inst V1 (E) Plus by Unwa' : 'melband_roformer_inst_v1e_plus.ckpt',
      'MelBand Roformer Kim | Inst V2 by Unwa' : 'melband_roformer_inst_v2.ckpt',
      'MelBand Roformer Kim | InstVoc Duality V1 by Unwa' : 'melband_roformer_instvoc_duality_v1.ckpt',
      'MelBand Roformer Kim | InstVoc Duality V2 by Unwa' : 'melband_roformer_instvox_duality_v2.ckpt',
      'MelBand Roformer | Vocals by becruily' : 'mel_band_roformer_vocals_becruily.ckpt',
      'MelBand Roformer | Instrumental by becruily' : 'mel_band_roformer_instrumental_becruily.ckpt',
      'MelBand Roformer | Vocals Fullness by Aname' : 'mel_band_roformer_vocal_fullness_aname.ckpt',
      'BS Roformer | Vocals by Gabox' : 'bs_roformer_vocals_gabox.ckpt',
      'MelBand Roformer | Vocals by Gabox' : 'mel_band_roformer_vocals_gabox.ckpt',
      'MelBand Roformer | Vocals V2 by Gabox' : 'mel_band_roformer_vocals_v2_gabox.ckpt',
      'MelBand Roformer | Vocals FV1 by Gabox' : 'mel_band_roformer_vocals_fv1_gabox.ckpt',
      'MelBand Roformer | Vocals FV2 by Gabox' : 'mel_band_roformer_vocals_fv2_gabox.ckpt',
      'MelBand Roformer | Vocals FV3 by Gabox' : 'mel_band_roformer_vocals_fv3_gabox.ckpt',
      'MelBand Roformer | Vocals FV4 by Gabox' : 'mel_band_roformer_vocals_fv4_gabox.ckpt',
      'MelBand Roformer | Vocals FV5 by Gabox' : 'mel_band_roformer_vocals_fv5_gabox.ckpt',
      'MelBand Roformer | Vocals FV6 by Gabox' : 'mel_band_roformer_vocals_fv6_gabox.ckpt',
      'MelBand Roformer | Instrumental by Gabox' : 'mel_band_roformer_instrumental_gabox.ckpt',
      'MelBand Roformer | Instrumental 2 by Gabox' : 'mel_band_roformer_instrumental_2_gabox.ckpt',
      'MelBand Roformer | Instrumental 3 by Gabox' : 'mel_band_roformer_instrumental_3_gabox.ckpt',
      'MelBand Roformer | Instrumental Bleedless V1 by Gabox' : 'mel_band_roformer_instrumental_bleedless_v1_gabox.ckpt',
      'MelBand Roformer | Instrumental Bleedless V2 by Gabox' : 'mel_band_roformer_instrumental_bleedless_v2_gabox.ckpt',
      'MelBand Roformer | Instrumental Bleedless V3 by Gabox' : 'mel_band_roformer_instrumental_bleedless_v3_gabox.ckpt',
      'MelBand Roformer | Instrumental Fullness V1 by Gabox' : 'mel_band_roformer_instrumental_fullness_v1_gabox.ckpt',
      'MelBand Roformer | Instrumental Fullness V2 by Gabox' : 'mel_band_roformer_instrumental_fullness_v2_gabox.ckpt',
      'MelBand Roformer | Instrumental Fullness V3 by Gabox' : 'mel_band_roformer_instrumental_fullness_v3_gabox.ckpt',
      'MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox' : 'mel_band_roformer_instrumental_fullness_noise_v4_gabox.ckpt',
      'MelBand Roformer | INSTV5 by Gabox' : 'mel_band_roformer_instrumental_instv5_gabox.ckpt',
      'MelBand Roformer | INSTV5N by Gabox' : 'mel_band_roformer_instrumental_instv5n_gabox.ckpt',
      'MelBand Roformer | INSTV6 by Gabox' : 'mel_band_roformer_instrumental_instv6_gabox.ckpt',
      'MelBand Roformer | INSTV6N by Gabox' : 'mel_band_roformer_instrumental_instv6n_gabox.ckpt',
      'MelBand Roformer | INSTV7 by Gabox' : 'mel_band_roformer_instrumental_instv7_gabox.ckpt',
      'MelBand Roformer | INSTV7N by Gabox' : 'mel_band_roformer_instrumental_instv7n_gabox.ckpt',
      'MelBand Roformer | INSTV8 by Gabox' : 'mel_band_roformer_instrumental_instv8_gabox.ckpt',
      'MelBand Roformer | INSTV8N by Gabox' : 'mel_band_roformer_instrumental_instv8n_gabox.ckpt',
      'MelBand Roformer | Instrumental FV7z by Gabox' : 'mel_band_roformer_instrumental_fv7z_gabox.ckpt',
      'MelBand Roformer | Instrumental FV8 by Gabox' : 'mel_band_roformer_instrumental_fv8_gabox.ckpt',
      'MelBand Roformer | Instrumental FVX by Gabox' : 'mel_band_roformer_instrumental_fvx_gabox.ckpt',
      'MelBand Roformer | De-Reverb by anvuew' : 'dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt',
      'MelBand Roformer | De-Reverb Less Aggressive by anvuew' : 'dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt',
      'MelBand Roformer | De-Reverb Mono by anvuew' : 'dereverb_mel_band_roformer_mono_anvuew.ckpt',
      'MelBand Roformer | De-Reverb Big by Sucial' : 'dereverb_big_mbr_ep_362.ckpt',
      'MelBand Roformer | De-Reverb Super Big by Sucial' : 'dereverb_super_big_mbr_ep_346.ckpt',
      'MelBand Roformer | De-Reverb-Echo by Sucial' : 'dereverb-echo_mel_band_roformer_sdr_10.0169.ckpt',
      'MelBand Roformer | De-Reverb-Echo V2 by Sucial' : 'dereverb-echo_mel_band_roformer_sdr_13.4843_v2.ckpt',
      'MelBand Roformer | De-Reverb-Echo Fused by Sucial' : 'dereverb_echo_mbr_fused.ckpt',
      'MelBand Roformer Kim | SYHFT by SYH99999' : 'MelBandRoformerSYHFT.ckpt',
      'MelBand Roformer Kim | SYHFT V2 by SYH99999' : 'MelBandRoformerSYHFTV2.ckpt',
      'MelBand Roformer Kim | SYHFT V2.5 by SYH99999' : 'MelBandRoformerSYHFTV2.5.ckpt',
      'MelBand Roformer Kim | SYHFT V3 by SYH99999' : 'MelBandRoformerSYHFTV3Epsilon.ckpt',
      'MelBand Roformer Kim | Big SYHFT V1 by SYH99999' : 'MelBandRoformerBigSYHFTV1.ckpt',
      'MelBand Roformer Kim | Big Beta 4 FT by unwa' : 'melband_roformer_big_beta4.ckpt',
      'MelBand Roformer Kim | Big Beta 5e FT by unwa' : 'melband_roformer_big_beta5e.ckpt',
      'MelBand Roformer | Big Beta 6 by unwa' : 'melband_roformer_big_beta6.ckpt',
      'MelBand Roformer | Big Beta 6X by unwa' : 'melband_roformer_big_beta6x.ckpt',
      'BS Roformer | Vocals Revive by Unwa' : 'bs_roformer_vocals_revive_unwa.ckpt',
      'BS Roformer | Vocals Revive V2 by Unwa' : 'bs_roformer_vocals_revive_v2_unwa.ckpt',
      'BS Roformer | Vocals Revive V3e by Unwa' : 'bs_roformer_vocals_revive_v3e_unwa.ckpt',
      'BS Roformer | Chorus Male-Female by Sucial' : 'model_chorus_bs_roformer_ep_267_sdr_24.1275.ckpt',
      'BS Roformer | Male-Female by aufr33' : 'bs_roformer_male_female_by_aufr33_sdr_7.2889.ckpt',
      'MelBand Roformer | Aspiration by Sucial' : 'aspiration_mel_band_roformer_sdr_18.9845.ckpt',
      'MelBand Roformer | Aspiration Less Aggressive by Sucial' : 'aspiration_mel_band_roformer_less_aggr_sdr_18.1201.ckpt',
      'MelBand Roformer | Bleed Suppressor V1 by unwa-97chris' : 'mel_band_roformer_bleed_suppressor_v1.ckpt',
      'BS Roformer | Vocals Resurrection by unwa' : 'bs_roformer_vocals_resurrection_unwa.ckpt',
      'BS Roformer | Instrumental Resurrection by unwa' : 'bs_roformer_instrumental_resurrection_unwa.ckpt'
    }

    roformer_model = dictmodel[model]

    if checker(audio_input):
        downloader(audio_input)
        audio_input = f"{path}/temp"

    for audio_files in os.listdir(audio_input):
        if audio_files.endswith(extensions):
            found_files.append(audio_files)

    total_files = len(found_files)

    if total_files == 0:
        print("有効な音声ファイルが見つかりませんでした。")
    else:
        print(f"{total_files} 件の音声ファイルが見つかりました")

        found_files.sort()

        for audio_files in found_files:
            file_path = os.path.join(audio_input, audio_files)
            prompt = f'audio-separator "{file_path}" --model_filename {roformer_model} --output_dir={output_folder} --output_format={output_format} --mdxc_segment_size={segment_size} --mdxc_overlap={overlap} --mdxc_batch_size={batch_size} --model_file_dir=./models'
            if override_model_segment_size:
                prompt += " --mdxc_override_model_segment_size"
            if use_autocast:
                prompt += " --use_autocast"
            !$prompt

    if audio_input == f"{path}/temp":
        temp_files = glob.glob(f"{path}/temp/*")
        for file in temp_files:
            os.remove(file)

#@markdown 音声ファイルの入力パス（フォルダ）またはリンク：
audio_input = "ここにパスまたはリンクを入力" #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイルの出力先パス：
output_folder = "/content/drive/MyDrive/Vocales" #@param {type:"string"}
#@markdown モデルを選択：<br>※ カッコ内は用途の目安です（audio-separator 収録のモデル情報とモデル名に基づく）
model = "BS-Roformer-Viperx-1297（ボーカル／インスト）" #@param ["BS-Roformer-Viperx-1297（ボーカル／インスト）", "BS-Roformer-Viperx-1296（ボーカル／インスト）", "BS-Roformer-Viperx-1053（ドラム+ベース／それ以外）", "Mel-Roformer-Viperx-1143（ボーカル／インスト）", "BS-Roformer-De-Reverb（リバーブ除去）", "Mel-Roformer-Crowd-Aufr33-Viperx（観客音の分離）", "Mel-Roformer-Denoise-Aufr33（ノイズ除去）", "Mel-Roformer-Denoise-Aufr33-Aggr（ノイズ除去）", "MelBand Roformer | Denoise-Debleed by Gabox（ノイズ・音漏れ除去）", "Mel-Roformer-Karaoke-Aufr33-Viperx（カラオケ）", "MelBand Roformer | Karaoke by Gabox（カラオケ）", "MelBand Roformer | Karaoke V2 by Gabox（カラオケ）", "MelBand Roformer | Karaoke by becruily（カラオケ）", "MelBand Roformer | Vocals by Kimberley Jensen（ボーカル／その他）", "MelBand Roformer Kim | FT by unwa（ボーカル／その他）", "MelBand Roformer Kim | FT 2 by unwa", "MelBand Roformer Kim | FT 2 Bleedless by unwa", "MelBand Roformer Kim | FT 3 by unwa", "MelBand Roformer Kim | Inst V1 by Unwa（ボーカル／インスト）", "MelBand Roformer Kim | Inst V1 Plus by Unwa（インスト）", "MelBand Roformer Kim | Inst V1 (E) by Unwa（ボーカル／インスト）", "MelBand Roformer Kim | Inst V1 (E) Plus by Unwa（インスト）", "MelBand Roformer Kim | Inst V2 by Unwa（ボーカル／インスト）", "MelBand Roformer Kim | InstVoc Duality V1 by Unwa（ボーカル／インスト）", "MelBand Roformer Kim | InstVoc Duality V2 by Unwa（ボーカル／インスト）", "MelBand Roformer | Vocals by becruily（ボーカル）", "MelBand Roformer | Instrumental by becruily（インスト）", "MelBand Roformer | Vocals Fullness by Aname（ボーカル）", "BS Roformer | Vocals by Gabox（ボーカル）", "MelBand Roformer | Vocals by Gabox（ボーカル）", "MelBand Roformer | Vocals V2 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV1 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV2 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV3 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV4 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV5 by Gabox（ボーカル）", "MelBand Roformer | Vocals FV6 by Gabox（ボーカル）", "MelBand Roformer | Instrumental by Gabox（インスト）", "MelBand Roformer | Instrumental 2 by Gabox（インスト）", "MelBand Roformer | Instrumental 3 by Gabox（インスト）", "MelBand Roformer | Instrumental Bleedless V1 by Gabox（インスト）", "MelBand Roformer | Instrumental Bleedless V2 by Gabox（インスト）", "MelBand Roformer | Instrumental Bleedless V3 by Gabox（インスト）", "MelBand Roformer | Instrumental Fullness V1 by Gabox（インスト）", "MelBand Roformer | Instrumental Fullness V2 by Gabox（インスト）", "MelBand Roformer | Instrumental Fullness V3 by Gabox（インスト）", "MelBand Roformer | Instrumental Fullness Noisy V4 by Gabox（インスト）", "MelBand Roformer | INSTV5 by Gabox（インスト）", "MelBand Roformer | INSTV5N by Gabox（インスト）", "MelBand Roformer | INSTV6 by Gabox（インスト）", "MelBand Roformer | INSTV6N by Gabox（インスト）", "MelBand Roformer | INSTV7 by Gabox（インスト）", "MelBand Roformer | INSTV7N by Gabox（インスト）", "MelBand Roformer | INSTV8 by Gabox（インスト）", "MelBand Roformer | INSTV8N by Gabox（インスト）", "MelBand Roformer | Instrumental FV7z by Gabox（インスト）", "MelBand Roformer | Instrumental FV8 by Gabox（インスト）", "MelBand Roformer | Instrumental FVX by Gabox（インスト）", "MelBand Roformer | De-Reverb by anvuew（リバーブ除去）", "MelBand Roformer | De-Reverb Less Aggressive by anvuew（リバーブ除去）", "MelBand Roformer | De-Reverb Mono by anvuew（リバーブ除去）", "MelBand Roformer | De-Reverb Big by Sucial（リバーブ除去）", "MelBand Roformer | De-Reverb Super Big by Sucial（リバーブ除去）", "MelBand Roformer | De-Reverb-Echo by Sucial（リバーブ・エコー除去）", "MelBand Roformer | De-Reverb-Echo V2 by Sucial（リバーブ・エコー除去）", "MelBand Roformer | De-Reverb-Echo Fused by Sucial（リバーブ・エコー除去）", "MelBand Roformer Kim | SYHFT by SYH99999（ボーカル／その他）", "MelBand Roformer Kim | SYHFT V2 by SYH99999（ボーカル／その他）", "MelBand Roformer Kim | SYHFT V2.5 by SYH99999（ボーカル／その他）", "MelBand Roformer Kim | SYHFT V3 by SYH99999（ボーカル／その他）", "MelBand Roformer Kim | Big SYHFT V1 by SYH99999（ボーカル／その他）", "MelBand Roformer Kim | Big Beta 4 FT by unwa（ボーカル／その他）", "MelBand Roformer Kim | Big Beta 5e FT by unwa（ボーカル／その他）", "MelBand Roformer | Big Beta 6 by unwa", "MelBand Roformer | Big Beta 6X by unwa", "BS Roformer | Vocals Revive by Unwa（ボーカル）", "BS Roformer | Vocals Revive V2 by Unwa（ボーカル）", "BS Roformer | Vocals Revive V3e by Unwa（ボーカル）", "BS Roformer | Chorus Male-Female by Sucial（コーラスの男声／女声）", "BS Roformer | Male-Female by aufr33（男声／女声）", "MelBand Roformer | Aspiration by Sucial（ブレス音の分離）", "MelBand Roformer | Aspiration Less Aggressive by Sucial（ブレス音の分離）", "MelBand Roformer | Bleed Suppressor V1 by unwa-97chris（インストの音漏れ除去）", "BS Roformer | Vocals Resurrection by unwa（ボーカル）", "BS Roformer | Instrumental Resurrection by unwa（インスト）"]
#@markdown 出力形式を選択：
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown 大きいほどリソース消費が増えますが、より良い結果になる場合があります。
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown 予測ウィンドウ間の重なり（オーバーラップ）量。
overlap = 8 #@param {type:"slider", min:2, max:10, step:1}
#@markdown 大きいほど RAM 消費が増えますが、処理がやや速くなる場合があります。
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown モデル既定のセグメントサイズを使わず、上で指定した値で上書きします。
override_model_segment_size = False #@param {type:"boolean"}
#@markdown PyTorch の autocast を使用して推論を高速化します。CPU で推論する場合は使用しないでください。
use_autocast = True #@param {type:"boolean"}
extensions = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3")

model = model.split("（")[0]  # 用途の補足を除き、元のモデル名に戻す

uvr_cli(audio_input, output_folder, model, output_format, segment_size, overlap, batch_size, override_model_segment_size, use_autocast, extensions)

### ***MDX23C***

In [ ]:
#@markdown #**分離実行！（MDX23C 専用）**
import os
import glob
import yt_dlp

def downloader(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

def checker(url):
    return "http" in url

def uvr_cli(audio_input, output_folder, model, output_format, segment_size, overlap, batch_size, override_model_segment_size, use_autocast, extensions):
    found_files = []

    if checker(audio_input):
        downloader(audio_input)
        audio_input = f"{path}/temp"

    for audio_files in os.listdir(audio_input):
        if audio_files.endswith(extensions):
            found_files.append(audio_files)

    total_files = len(found_files)

    if total_files == 0:
        print("有効な音声ファイルが見つかりませんでした。")
    else:
        print(f"{total_files} 件の音声ファイルが見つかりました")

        found_files.sort()

        for audio_files in found_files:
            file_path = os.path.join(audio_input, audio_files)
            prompt = f'audio-separator "{file_path}" --model_filename {model} --output_dir={output_folder} --output_format={output_format} --mdxc_segment_size={segment_size} --mdxc_overlap={overlap} --mdxc_batch_size={batch_size} --model_file_dir=./models'
            if override_model_segment_size:
                prompt += " --mdxc_override_model_segment_size"
            if use_autocast:
                prompt += " --use_autocast"
            !$prompt

    if audio_input == f"{path}/temp":
        temp_files = glob.glob(f"{path}/temp/*")
        for file in temp_files:
            os.remove(file)

#@markdown 音声ファイルの入力パス（フォルダ）またはリンク：
audio_input = "ここにパスまたはリンクを入力" #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイルの出力先パス：
output_folder = "/content/drive/MyDrive/Vocales" #@param {type:"string"}
#@markdown モデルを選択：<br>※ カッコ内は用途の目安です（audio-separator 収録のモデル情報とモデル名に基づく）
model = "MDX23C-8KFFT-InstVoc_HQ_2.ckpt（ボーカル／インスト）" #@param ["MDX23C_D1581.ckpt（ボーカル／インスト）", "MDX23C-8KFFT-InstVoc_HQ.ckpt（ボーカル／インスト）", "MDX23C-8KFFT-InstVoc_HQ_2.ckpt（ボーカル／インスト）", "MDX23C-De-Reverb-aufr33-jarredou.ckpt（リバーブ除去）", "MDX23C-DrumSep-aufr33-jarredou.ckpt（ドラムのパーツ分離）"]
#@markdown 出力形式を選択：
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown 大きいほどリソース消費が増えますが、より良い結果になる場合があります。
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown 予測ウィンドウ間の重なり（オーバーラップ）量。
overlap = 8 #@param {type:"slider", min:2, max:50, step:1}
#@markdown 大きいほど RAM 消費が増えますが、処理がやや速くなる場合があります。
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown モデル既定のセグメントサイズを使わず、上で指定した値で上書きします。
override_model_segment_size = False #@param {type:"boolean"}
#@markdown PyTorch の autocast を使用して推論を高速化します。CPU で推論する場合は使用しないでください。
use_autocast = True #@param {type:"boolean"}
extensions = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3")

model = model.split("（")[0]  # 用途の補足を除き、元のモデル名に戻す

uvr_cli(audio_input, output_folder, model, output_format, segment_size, overlap, batch_size, override_model_segment_size, use_autocast, extensions)

### ***MDX-NET***

In [ ]:
#@markdown #**分離実行！（MDX-NET 専用）**
import os
import glob
import yt_dlp

def downloader(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

def checker(url):
    return "http" in url

def uvr_cli(audio_input, output_folder, extensions, output_format, model, hop_length, segment_size, denoise, overlap, batch_size, use_autocast):
    found_files = []

    if checker(audio_input):
        downloader(audio_input)
        audio_input = f"{path}/temp"

    for audio_files in os.listdir(audio_input):
        if audio_files.endswith(extensions):
            found_files.append(audio_files)

    total_files = len(found_files)

    if total_files == 0:
        print("有効な音声ファイルが見つかりませんでした。")
    else:
        print(f"{total_files} 件の音声ファイルが見つかりました")

        found_files.sort()

        for audio_files in found_files:
            file_path = os.path.join(audio_input, audio_files)
            prompt = f'audio-separator "{file_path}" --model_filename {model} --output_dir={output_folder} --output_format={output_format} --mdx_hop_length={hop_length} --mdx_segment_size={segment_size} --mdx_overlap={overlap} --mdx_batch_size={batch_size} --model_file_dir=./models'
            if denoise:
                prompt += " --mdx_enable_denoise"
            if use_autocast:
                prompt += " --use_autocast"
            !$prompt

    if audio_input == f"{path}/temp":
        temp_files = glob.glob(f"{path}/temp/*")
        for file in temp_files:
            os.remove(file)

#@markdown 音声ファイルの入力パス（フォルダ）またはリンク：
audio_input = "ここにパスまたはリンクを入力" #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイルの出力先パス：
output_folder = "/content/drive/MyDrive/Vocales" #@param {type:"string"}
#@markdown モデルを選択：<br>※ カッコ内は用途の目安です（audio-separator 収録のモデル情報とモデル名に基づく）
model = "UVR-MDX-NET-Inst_HQ_5.onnx（ボーカル／インスト）" #@param ["UVR-MDX-NET-Inst_full_292.onnx（ボーカル／インスト）", "UVR-MDX-NET_Inst_187_beta.onnx（ボーカル／インスト）", "UVR-MDX-NET_Inst_82_beta.onnx（ボーカル／インスト）", "UVR-MDX-NET_Inst_90_beta.onnx（ボーカル／インスト）", "UVR-MDX-NET_Main_340.onnx（ボーカル／インスト）", "UVR-MDX-NET_Main_390.onnx（ボーカル／インスト）", "UVR-MDX-NET_Main_406.onnx（ボーカル／インスト）", "UVR-MDX-NET_Main_427.onnx（ボーカル／インスト）", "UVR-MDX-NET_Main_438.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_HQ_1.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_HQ_2.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_HQ_3.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_HQ_4.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_HQ_5.onnx（ボーカル／インスト）", "UVR_MDXNET_Main.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_Main.onnx（ボーカル／インスト）", "UVR_MDXNET_1_9703.onnx（ボーカル／インスト）", "UVR_MDXNET_2_9682.onnx（ボーカル／インスト）", "UVR_MDXNET_3_9662.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_1.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_2.onnx（ボーカル／インスト）", "UVR-MDX-NET-Inst_3.onnx（ボーカル／インスト）", "UVR_MDXNET_KARA.onnx（カラオケ）", "UVR_MDXNET_KARA_2.onnx（カラオケ）", "UVR_MDXNET_9482.onnx（ボーカル／インスト）", "UVR-MDX-NET-Voc_FT.onnx（ボーカル／インスト）", "Kim_Vocal_1.onnx（ボーカル／インスト）", "Kim_Vocal_2.onnx（ボーカル／インスト）", "Kim_Inst.onnx（ボーカル／インスト）", "Reverb_HQ_By_FoxJoy.onnx（リバーブ除去）", "UVR-MDX-NET_Crowd_HQ_1.onnx（観客音の分離）", "kuielab_a_vocals.onnx（ボーカル／インスト）", "kuielab_a_other.onnx（その他の楽器）", "kuielab_a_bass.onnx（ベース）", "kuielab_a_drums.onnx（ドラム）", "kuielab_b_vocals.onnx（ボーカル／インスト）", "kuielab_b_other.onnx（その他の楽器）", "kuielab_b_bass.onnx（ベース）", "kuielab_b_drums.onnx（ドラム）"]
#@markdown 出力形式を選択：
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown ニューラルネットワークでは一般に「ストライド」と呼ばれる値です。理解している場合のみ変更してください。
hop_length = 1024 #@param {type:"slider", min:32, max:2048, step:32}
#@markdown 大きいほどリソース消費が増えますが、より良い結果になる場合があります。
segment_size = 256 #@param {type:"slider", min:32, max:4000, step:32}
#@markdown 予測ウィンドウ間の重なり（オーバーラップ）量。
overlap = "0.25" #@param ["0.25", "0.5", "0.75", "0.99"]
#@markdown 大きいほど RAM 消費が増えますが、処理がやや速くなる場合があります。
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown 分離時にノイズ除去を有効にします。
denoise = True #@param {type:"boolean"}
#@markdown PyTorch の autocast を使用して推論を高速化します。CPU で推論する場合は使用しないでください。
use_autocast = True #@param {type:"boolean"}
extensions = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3")

model = model.split("（")[0]  # 用途の補足を除き、元のモデル名に戻す

uvr_cli(audio_input, output_folder, extensions, output_format, model, hop_length, segment_size, denoise, overlap, batch_size, use_autocast)

### ***VR ARCH***

In [ ]:
#@markdown #**分離実行！（VR ARCH 専用）**
import os
import glob
import yt_dlp

def downloader(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

def checker(url):
    return "http" in url

def uvr_cli(audio_input, output_folder, extensions, output_format, model, window_size, aggression, tta , high_end_process, batch_size, use_autocast):
    found_files = []

    if checker(audio_input):
        downloader(audio_input)
        audio_input = f"{path}/temp"

    for audio_files in os.listdir(audio_input):
        if audio_files.endswith(extensions):
            found_files.append(audio_files)

    total_files = len(found_files)

    if total_files == 0:
        print("有効な音声ファイルが見つかりませんでした。")
    else:
        print(f"{total_files} 件の音声ファイルが見つかりました")

        found_files.sort()

        for audio_files in found_files:
            file_path = os.path.join(audio_input, audio_files)
            prompt = f'audio-separator "{file_path}" --model_filename {model} --output_dir={output_folder} --output_format={output_format} --vr_window_size={window_size} --vr_aggression={aggression} --vr_batch_size={batch_size} --model_file_dir=./models'
            if tta:
              prompt += " --vr_enable_tta"
            if high_end_process:
              prompt += " --vr_high_end_process"
            if use_autocast:
              prompt += " --use_autocast"
            !$prompt

    if audio_input == f"{path}/temp":
        temp_files = glob.glob(f"{path}/temp/*")
        for file in temp_files:
            os.remove(file)

#@markdown 音声ファイルの入力パス（フォルダ）またはリンク：
audio_input = "ここにパスまたはリンクを入力" #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイルの出力先パス：
output_folder = "/content/drive/MyDrive/Vocales" #@param {type:"string"}
#@markdown モデルを選択：<br>※ カッコ内は用途の目安です（audio-separator 収録のモデル情報とモデル名に基づく）
model = "UVR-DeEcho-DeReverb.pth（エコー・リバーブ除去）" #@param ["1_HP-UVR.pth（ボーカル／インスト）", "2_HP-UVR.pth（ボーカル／インスト）", "3_HP-Vocal-UVR.pth（ボーカル／インスト）", "4_HP-Vocal-UVR.pth（ボーカル／インスト）", "5_HP-Karaoke-UVR.pth（カラオケ）", "6_HP-Karaoke-UVR.pth（カラオケ）", "7_HP2-UVR.pth（ボーカル／インスト）", "8_HP2-UVR.pth（ボーカル／インスト）", "9_HP2-UVR.pth（ボーカル／インスト）", "10_SP-UVR-2B-32000-1.pth（ボーカル／インスト）", "11_SP-UVR-2B-32000-2.pth（ボーカル／インスト）", "12_SP-UVR-3B-44100.pth（ボーカル／インスト）", "13_SP-UVR-4B-44100-1.pth（ボーカル／インスト）", "14_SP-UVR-4B-44100-2.pth（ボーカル／インスト）", "15_SP-UVR-MID-44100-1.pth（ボーカル／インスト）", "16_SP-UVR-MID-44100-2.pth（ボーカル／インスト）", "17_HP-Wind_Inst-UVR.pth（木管楽器）", "UVR-De-Echo-Aggressive.pth（エコー除去）", "UVR-De-Echo-Normal.pth（エコー除去）", "UVR-DeEcho-DeReverb.pth（エコー・リバーブ除去）", "UVR-De-Reverb-aufr33-jarredou.pth（リバーブ除去）", "UVR-DeNoise-Lite.pth（ノイズ除去）", "UVR-DeNoise.pth（ノイズ除去）", "UVR-BVE-4B_SN-44100-1.pth", "UVR-BVE-4B_SN-44100-2.pth", "MGM_HIGHEND_v4.pth（ボーカル／インスト）", "MGM_LOWEND_A_v4.pth（ボーカル／インスト）", "MGM_LOWEND_B_v4.pth（ボーカル／インスト）", "MGM_MAIN_v4.pth（ボーカル／インスト）"]
#@markdown 出力形式を選択：
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown ウィンドウサイズが小さいほど出力品質は向上しますが、処理時間が長くなります。
window_size = 512 #@param {type:"slider", min:320, max:1024, step:32}
#@markdown メインステム抽出の強度。
aggression = 5 #@param {type:"slider", min:1, max:50, step:1}
#@markdown 大きいほど RAM 消費が増えますが、処理がやや速くなる場合があります。
batch_size = 1 #@param {type:"slider", min:1, max:16, step:1}
#@markdown TTA（Test-Time Augmentation）を有効にします。処理は遅くなりますが品質が向上します。
tta = True #@param {type:"boolean"}
#@markdown 出力で欠落した高域の周波数帯をミラーリングして補完します。
high_end_process = False #@param {type:"boolean"}
#@markdown PyTorch の autocast を使用して推論を高速化します。CPU で推論する場合は使用しないでください。
use_autocast = True #@param {type:"boolean"}
extensions = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3")

model = model.split("（")[0]  # 用途の補足を除き、元のモデル名に戻す

uvr_cli(audio_input, output_folder, extensions, output_format, model, window_size, aggression, tta , high_end_process, batch_size, use_autocast)

### ***Demucs***

In [ ]:
#@markdown #**分離実行！（Demucs 専用）**
import os
import glob
import yt_dlp

def downloader(url):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(f'{path}/temp', '%(title)s.%(ext)s'),
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

def checker(url):
    return "http" in url

def uvr_cli(audio_input, output_folder, extensions, output_format, model, shifts, segment_size, overlap, use_autocast):
    found_files = []

    if checker(audio_input):
        downloader(audio_input)
        audio_input = f"{path}/temp"

    for audio_files in os.listdir(audio_input):
        if audio_files.endswith(extensions):
            found_files.append(audio_files)

    total_files = len(found_files)

    if total_files == 0:
        print("有効な音声ファイルが見つかりませんでした。")
    else:
        print(f"{total_files} 件の音声ファイルが見つかりました")

        found_files.sort()

        for audio_files in found_files:
            file_path = os.path.join(audio_input, audio_files)
            if model == "drumsep":
              print(f"処理中のファイル: {audio_files}")
              prompt = f'demucs --repo {path}/models --shifts={shifts} --overlap={overlap} -o {output_folder} -n drumsep "{file_path}"'
              !$prompt
              print(f"ファイル: {audio_files} の処理が完了しました！")
            else:
              prompt = f'audio-separator "{file_path}" --model_filename {model} --output_dir={output_folder} --output_format={output_format} --demucs_shifts={shifts} --demucs_overlap={overlap} --demucs_segment_size={segment_size} --model_file_dir=./models'
              if use_autocast:
                  prompt += " --use_autocast"
              !$prompt

    if audio_input == f"{path}/temp":
        temp_files = glob.glob(f"{path}/temp/*")
        for file in temp_files:
            os.remove(file)

#@markdown 音声ファイルの入力パス（フォルダ）またはリンク：
audio_input = "ここにパスまたはリンクを入力" #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイルの出力先パス：
output_folder = "/content/drive/MyDrive/Vocales" #@param {type:"string"}
#@markdown モデルを選択：<br>※ カッコ内は用途の目安です（audio-separator 収録のモデル情報とモデル名に基づく）
model = "htdemucs_ft.yaml（4ステム：ボーカル・ドラム・ベース・その他）" #@param ["drumsep（ドラム分離）", "htdemucs_ft.yaml（4ステム：ボーカル・ドラム・ベース・その他）", "htdemucs.yaml（4ステム：ボーカル・ドラム・ベース・その他）", "hdemucs_mmi.yaml（4ステム：ボーカル・ドラム・ベース・その他）", "htdemucs_6s.yaml（6ステム：ボーカル・ドラム・ベース・ギター・ピアノ・その他）"]
#@markdown 出力形式を選択：
output_format = "wav" #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]
#@markdown ランダムシフトによる予測回数。大きいほど遅くなりますが品質が向上します。
shifts = 2 #@param {type:"slider", min:1, max:20, step:1}
#@markdown 予測ウィンドウ間の重なり（オーバーラップ）量。
overlap = 0.025 #@param {type:"slider", min:0.001, max:0.999, step:0.001}
#@markdown 大きいほどリソース消費が増えますが、より良い結果になる場合があります。
segment_size = 40 #@param {type:"slider", min:1, max:100, step:1}
#@markdown PyTorch の autocast を使用して推論を高速化します。CPU で推論する場合は使用しないでください。
use_autocast = True #@param {type:"boolean"}
extensions = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a", ".aiff", ".ac3")

model = model.split("（")[0]  # 用途の補足を除き、元のモデル名に戻す

uvr_cli(audio_input, output_folder, extensions, output_format, model, shifts, segment_size, overlap, use_autocast)

# <small> **おまけ機能**
ベースコード：[Blane187](https://github.com/Blane187)

In [ ]:
#@markdown #**分離用オーディオダウンローダー**
import os
import yt_dlp

#@markdown リンク：
video_url = "https://youtu.be/L3P0LgwuPPo?si=8uQWCX9kiXgMxYdE"  #@param {type:"string"}
#@markdown 多くのサイトの動画・音声リンクを貼り付けられます。対応サイトの一覧は[こちら](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md)

#@markdown 音声ファイル名：
audio_name = "Harenchi"  #@param {type:"string"}
#@markdown ダウンロードした音声ファイルの保存先パス：
save_folder = "/content/drive/MyDrive/Separar"  #@param {type:"string"}
#@markdown 出力形式を選択：
audio_format = "wav"  #@param ["wav", "flac", "mp3", "ogg", "opus", "m4a", "aiff", "ac3"]

def downloader(url, save_path, audio_format, audio_name):
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': audio_format,
            'preferredquality': '192',
        }],
        'outtmpl': save_path,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

save_path = os.path.join(save_folder, audio_name)
downloader(video_url, save_path, audio_format, audio_name)
print("ダウンロードが完了しました！")